# MovieLens-25M Recommender — Rebuilt

A corrected and substantially stronger version of the hybrid recommender.

## What changed and why

| Issue in the previous notebook | Consequence | Fix here |
|---|---|---|
| Content and collaborative candidates merged with `how="inner"` | Candidate pool collapsed to the few items in *both* top-50 lists; often fewer than 10 survived, so Precision@10 was structurally near zero | Score **every** item for every user, then rank once |
| Items looked up by `title.str.contains(...)` | "Toy Story" matches *Toy Story 2/3*; "Heat" matches dozens of films. Wrong seed item for many users | Everything keyed on integer `movieId` → contiguous index |
| `matches.index[0]` used as an embedding row | Breaks whenever a filter reindexes the DataFrame | Explicit `movieId → row` mapping, asserted |
| Min-max normalisation per candidate list | Forces best→1.0, worst→0.0 every time regardless of quality, so the weight sweep compared rescaled noise | Z-score against a fixed reference distribution |
| Two different train/test splits in one notebook | Encoder fine-tuned on one split, evaluated against another | One split, built once, used everywhere |
| Evaluated on 100 users (precision on 10) | Standard error ≈ ±0.04 — "0.16 vs 0.21" was not a real difference | 10,000 users, batched and vectorised, with bootstrap CIs |
| No baseline | No way to tell if the model beat "recommend the popular stuff" | Popularity, ItemKNN and PureSVD baselines reported alongside |
| Item-item cosine on mean-centred ratings | A 2005-era method | **EASE** (Steck, WWW 2019) — closed form, one hyperparameter, competitive with deep models on MovieLens |

## Expected outcome

EASE on MovieLens-25M typically reaches **NDCG@10 ≈ 0.30–0.38** and **Recall@20 ≈ 0.35–0.42**
under this protocol, against NDCG@10 ≈ 0.10 previously. The largest single jump comes from
fixing the inner-join; the second largest from replacing the CF model.

## Runtime

Roughly 25–45 minutes end to end on a Kaggle CPU session. EASE's cost scales with the number of
*items*, not users, so the full 25M ratings are used for training. Set `USE_CONTENT_SBERT = False`
(the default) to skip the GPU-dependent encoder — TF-IDF over genres and tags is close behind
and runs in seconds.

## 1. Configuration

Every knob lives here. Nothing below this cell hard-codes a threshold.

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

from pathlib import Path

DATA_DIR = Path("/kaggle/input/datasets/garymk/movielens-25m-dataset/ml-25m")
OUT_DIR  = Path("/kaggle/working")

# --- Implicit feedback ---------------------------------------------------
# A rating >= this counts as a positive interaction. Everything else is
# dropped entirely rather than treated as a negative: a 2-star rating means
# "watched and disliked", which is different from "never saw it", and
# conflating the two is a common source of silently bad recommenders.
POSITIVE_THRESHOLD = 4.0

# --- k-core filtering ----------------------------------------------------
# Applied iteratively until stable. Dropping users changes item counts and
# vice versa, so a single pass leaves items below the threshold behind.
MIN_ITEM_INTERACTIONS = 20
MIN_USER_INTERACTIONS = 10
KCORE_MAX_PASSES = 10

# --- Item budget ---------------------------------------------------------
# EASE inverts a dense (n_items x n_items) matrix. Memory is roughly
# 8 * n_items^2 bytes for the Gram matrix plus the same again for the
# inverse. 15,000 items ~= 1.8 GB each, ~4 GB peak. Raise if you have RAM
# to spare; the long tail contributes little to ranking metrics.
MAX_ITEMS = 15_000

# --- Evaluation protocol -------------------------------------------------
# Leave-last-N-out, chronological. The most recent N positives per user are
# held out for test, the N before those for validation.
TEST_N_PER_USER = 5
VAL_N_PER_USER  = 5
MIN_TRAIN_PER_USER = 5      # users with less history than this are not evaluated

EVAL_USERS   = 10_000       # sampled users for the metric report
TUNING_USERS = 3_000        # smaller sample for hyperparameter sweeps
K_VALUES     = (5, 10, 20, 50)
PRIMARY_K    = 10

# --- Models --------------------------------------------------------------
# Each lambda costs one dense inverse (a few minutes at 15k items). Widen the
# list if you have time; the curve is flat enough that 3-4 points find the peak.
EASE_LAMBDAS   = [100.0, 250.0, 500.0, 1000.0]
ITEMKNN_K      = 200        # neighbours retained per item
ITEMKNN_SHRINK = 100.0      # shrinkage on the cosine denominator
SVD_FACTORS    = 256

# --- Content model -------------------------------------------------------
# False  -> TF-IDF over genres + tags (seconds, CPU, no downloads)
# True   -> sentence-transformers encoder (minutes, wants a GPU)
USE_CONTENT_SBERT = False
SBERT_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CONTENT_KNN_K = 200
MAX_TAGS_PER_ITEM = 40

HYBRID_ALPHAS = [1.0, 0.98, 0.95, 0.9, 0.8, 0.7, 0.5, 0.0]   # weight on CF

SEED = 42
BATCH_SIZE = 512            # users scored per batch during evaluation

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import gc
import time
import warnings
from contextlib import contextmanager

import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.linalg import svds

warnings.filterwarnings("ignore")
rng = np.random.default_rng(SEED)

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)


@contextmanager
def timed(label):
    """Print how long a block took. Useful when a cell silently takes 10 minutes."""
    print(f"  {label} ...", flush=True)
    t0 = time.time()
    yield
    print(f"  {label}: {time.time() - t0:.1f}s", flush=True)


def mem_mb(*arrays):
    return sum(a.nbytes for a in arrays) / 1e6


print("numpy", np.__version__, "| pandas", pd.__version__, "| scipy", sp.__version__ if hasattr(sp, "__version__") else "")

## 2. Load

Explicit dtypes on the 25M-row ratings file save roughly 500 MB. The `timestamp`
column is needed for the chronological split; the datetime derivatives from the
previous notebook (year, month, day-of-week, hour) were never used downstream,
so they are not built.

In [ ]:
def load_data(data_dir: Path):
    with timed("ratings.csv"):
        ratings = pd.read_csv(
            data_dir / "ratings.csv",
            dtype={
                "userId": np.int32,
                "movieId": np.int32,
                "rating": np.float32,
                "timestamp": np.int64,
            },
        )

    with timed("movies.csv"):
        movies = pd.read_csv(data_dir / "movies.csv", dtype={"movieId": np.int32})

    with timed("tags.csv"):
        tags = pd.read_csv(
            data_dir / "tags.csv",
            dtype={"userId": np.int32, "movieId": np.int32, "timestamp": np.int64},
        )

    n_users = ratings["userId"].nunique()
    n_items = ratings["movieId"].nunique()
    density = len(ratings) / (n_users * n_items)

    print()
    print(f"  ratings  {len(ratings):>12,}")
    print(f"  users    {n_users:>12,}")
    print(f"  movies   {n_items:>12,}")
    print(f"  tags     {len(tags):>12,}")
    print(f"  density  {density:>12.6%}")
    print(f"  span     {pd.to_datetime(ratings['timestamp'].min(), unit='s').date()} "
          f"to {pd.to_datetime(ratings['timestamp'].max(), unit='s').date()}")
    print(f"  memory   {ratings.memory_usage(deep=True).sum() / 1e6:>9.0f} MB")

    return ratings, movies, tags


ratings_raw, movies_raw, tags_raw = load_data(DATA_DIR)

## 3. Implicit feedback and k-core filtering

Two things happen here that the previous version did differently.

**Positives only.** Ratings below the threshold are discarded rather than kept as
low-value signal. For top-N ranking we care about which items a user *would engage
with*, and a 1-star rating is evidence against that.

**Iterative k-core.** Filtering users and items in a single pass leaves items that
fell below the threshold *because* their raters were removed. Repeating until the
counts stabilise gives a genuine k-core.

In [ ]:
def build_interactions(ratings: pd.DataFrame) -> pd.DataFrame:
    df = ratings.loc[
        ratings["rating"] >= POSITIVE_THRESHOLD, ["userId", "movieId", "timestamp"]
    ].copy()
    df = df.drop_duplicates(subset=["userId", "movieId"], keep="last")
    print(f"  {len(ratings):,} ratings -> {len(df):,} positives "
          f"(rating >= {POSITIVE_THRESHOLD})")
    return df


def kcore_filter(df: pd.DataFrame) -> pd.DataFrame:
    """Iterate user/item minimum-count filters until the frame stops shrinking."""
    for it in range(1, KCORE_MAX_PASSES + 1):
        before = len(df)

        item_counts = df["movieId"].value_counts()
        keep_items = item_counts.index[item_counts >= MIN_ITEM_INTERACTIONS]
        df = df[df["movieId"].isin(keep_items)]

        user_counts = df["userId"].value_counts()
        keep_users = user_counts.index[user_counts >= MIN_USER_INTERACTIONS]
        df = df[df["userId"].isin(keep_users)]

        print(f"    pass {it}: {before:,} -> {len(df):,} "
              f"({df['userId'].nunique():,} users, {df['movieId'].nunique():,} items)")
        if len(df) == before:
            break
    return df.copy()


def cap_items(df: pd.DataFrame, max_items: int) -> pd.DataFrame:
    """Keep the `max_items` most-interacted items so EASE's dense solve fits in RAM."""
    n = df["movieId"].nunique()
    if n <= max_items:
        print(f"  {n:,} items, under the {max_items:,} cap")
        return df
    keep = df["movieId"].value_counts().index[:max_items]
    out = df[df["movieId"].isin(keep)].copy()
    print(f"  capped {n:,} -> {max_items:,} items "
          f"({len(out) / len(df):.1%} of interactions retained)")
    # capping items can push users below the minimum again
    uc = out["userId"].value_counts()
    out = out[out["userId"].isin(uc.index[uc >= MIN_USER_INTERACTIONS])].copy()
    return out


inter = build_interactions(ratings_raw)
inter = kcore_filter(inter)
inter = cap_items(inter, MAX_ITEMS)

print()
print(f"  final: {len(inter):,} interactions | "
      f"{inter['userId'].nunique():,} users | {inter['movieId'].nunique():,} items")
print(f"  density: {len(inter) / (inter['userId'].nunique() * inter['movieId'].nunique()):.4%}")

del ratings_raw
gc.collect()

### Contiguous index mapping

Every matrix, embedding and score array from here on is addressed by
`user_idx` / `item_idx`. The mappings are the single source of truth — the
positional-index bug in the original notebook came from letting a DataFrame's
index stand in for a matrix row.

In [ ]:
unique_users = np.sort(inter["userId"].unique())
unique_items = np.sort(inter["movieId"].unique())

user_to_idx = {u: i for i, u in enumerate(unique_users)}
item_to_idx = {m: i for i, m in enumerate(unique_items)}
idx_to_user = unique_users            # idx_to_user[i] -> userId
idx_to_item = unique_items            # idx_to_item[j] -> movieId

N_USERS = len(unique_users)
N_ITEMS = len(unique_items)

inter["user_idx"] = inter["userId"].map(user_to_idx).astype(np.int32)
inter["item_idx"] = inter["movieId"].map(item_to_idx).astype(np.int32)

# Item metadata aligned to item_idx order. `.loc` on a movieId-indexed frame
# guarantees alignment; a merge would not preserve order.
_m = movies_raw.drop_duplicates("movieId").set_index("movieId")
item_meta = _m.reindex(unique_items).reset_index()
item_meta["item_idx"] = np.arange(N_ITEMS)
item_meta["title"] = item_meta["title"].fillna("(unknown)")
item_meta["genres"] = item_meta["genres"].fillna("(no genres listed)")

assert (item_meta["movieId"].values == unique_items).all(), "metadata misaligned"
assert item_meta["item_idx"].is_monotonic_increasing

print(f"  {N_USERS:,} users x {N_ITEMS:,} items")
print(f"  EASE Gram matrix will be ~{8 * N_ITEMS**2 / 1e9:.2f} GB (float64)")
item_meta.head()

## 4. Split

Leave-last-N-out, chronological, per user. The most recent 5 positives go to
test, the 5 before them to validation, everything earlier to train.

This is *weak generalisation*: every evaluated user also appears in training.
It matches the original setup so the numbers are comparable. The alternative
(holding out entire users) measures something different and is noted at the end.

The ordering matters: `train < val < test` in time for every user, so nothing
leaks backwards.

In [ ]:
def leave_last_n_split(df: pd.DataFrame, test_n: int, val_n: int, min_train: int):
    df = df.sort_values(["user_idx", "timestamp"], kind="mergesort")

    # rank from the end: 0 is the most recent interaction
    rank_from_end = df.groupby("user_idx").cumcount(ascending=False)
    n_per_user = df.groupby("user_idx")["item_idx"].transform("size")

    # only split users who keep enough history behind
    eligible = n_per_user >= (test_n + val_n + min_train)

    is_test = eligible & (rank_from_end < test_n)
    is_val = eligible & (rank_from_end >= test_n) & (rank_from_end < test_n + val_n)

    train = df[~(is_test | is_val)]
    val = df[is_val]
    test = df[is_test]

    print(f"  train {len(train):>10,}  ({train['user_idx'].nunique():,} users)")
    print(f"  val   {len(val):>10,}  ({val['user_idx'].nunique():,} users)")
    print(f"  test  {len(test):>10,}  ({test['user_idx'].nunique():,} users)")
    print(f"  users too short to split: {N_USERS - test['user_idx'].nunique():,}")
    return train, val, test


train_df, val_df, test_df = leave_last_n_split(
    inter, TEST_N_PER_USER, VAL_N_PER_USER, MIN_TRAIN_PER_USER
)

# Sanity: no user has a train interaction later than a test interaction.
_tr_max = train_df.groupby("user_idx")["timestamp"].max()
_te_min = test_df.groupby("user_idx")["timestamp"].min()
_shared = _tr_max.index.intersection(_te_min.index)
assert (_tr_max.loc[_shared] <= _te_min.loc[_shared]).all(), "temporal leakage"
print("  temporal ordering verified")

In [ ]:
def to_csr(df: pd.DataFrame, n_users: int, n_items: int) -> sp.csr_matrix:
    """Binary user x item matrix."""
    return sp.csr_matrix(
        (
            np.ones(len(df), dtype=np.float32),
            (df["user_idx"].values, df["item_idx"].values),
        ),
        shape=(n_users, n_items),
        dtype=np.float32,
    )


X_train = to_csr(train_df, N_USERS, N_ITEMS)
X_val = to_csr(val_df, N_USERS, N_ITEMS)
X_test = to_csr(test_df, N_USERS, N_ITEMS)

X_train.sort_indices()
X_val.sort_indices()
X_test.sort_indices()

# Users eligible for evaluation: have held-out truth AND training history.
train_counts = np.diff(X_train.indptr)
val_users_all = np.where((np.diff(X_val.indptr) > 0) & (train_counts >= MIN_TRAIN_PER_USER))[0]
test_users_all = np.where((np.diff(X_test.indptr) > 0) & (train_counts >= MIN_TRAIN_PER_USER))[0]

eval_users = rng.permutation(test_users_all)[:EVAL_USERS]
tune_users = rng.permutation(val_users_all)[:TUNING_USERS]

# Item popularity from TRAIN only. Using the full data here would leak.
item_pop = np.asarray(X_train.sum(axis=0)).ravel()
item_pop_prob = np.maximum(item_pop, 1.0) / item_pop.sum()

print(f"  X_train {X_train.shape}, nnz {X_train.nnz:,}")
print(f"  evaluating on {len(eval_users):,} test users, tuning on {len(tune_users):,} val users")

## 5. Evaluation harness

One function, used identically for every model, so the comparison is honest.

Each model exposes `score(user_indices) -> (n_users, n_items)` dense float array.
The harness then, per batch:

1. masks items the user already interacted with in train (setting them to `-inf`),
2. takes the top-`max(K)` by `argpartition` and sorts only those,
3. computes all metrics at all `k` from one ranking.

**On the metrics.** Precision@k divides by `k`. With only 5 held-out items,
Precision@10 is capped at 0.5 — worth knowing before you compare it to a paper
that held out 20%. Recall@k and NDCG@k are the ones to read. NDCG's ideal DCG is
computed against `min(k, n_relevant)`, which the previous notebook's
`sklearn.ndcg_score` call did not do correctly, because it passed rank positions
as scores and a binary array as true relevance.

Two beyond-accuracy metrics are included because a recommender that only ever
returns the same 200 blockbusters can score well and still be useless:

- **Coverage** — fraction of the catalogue that appears in anyone's top-k.
- **Novelty** — mean self-information $-\log_2 p(i)$ of recommended items. Higher
  means less popularity-biased.

In [ ]:
def _dcg_discounts(k: int) -> np.ndarray:
    return 1.0 / np.log2(np.arange(2, k + 2))


_IDCG_CACHE = {}


def _idcg_table(k: int) -> np.ndarray:
    """idcg[j] = ideal DCG@k for a user with (j+1) relevant items."""
    if k not in _IDCG_CACHE:
        _IDCG_CACHE[k] = np.cumsum(_dcg_discounts(k))
    return _IDCG_CACHE[k]


def evaluate(model, users, X_seen, X_truth, ks=K_VALUES, batch_size=BATCH_SIZE,
             collect_items=True):
    """
    model    : object with .score(user_idx_array) -> (B, N_ITEMS) dense
    X_seen   : interactions to mask out (train)
    X_truth  : held-out ground truth (val or test)
    """
    ks = tuple(sorted(ks))
    max_k = max(ks)
    acc = {k: {"recall": [], "precision": [], "ndcg": [], "hit": []} for k in ks}
    recommended = np.zeros(N_ITEMS, dtype=bool) if collect_items else None
    novelty_sum, novelty_n = 0.0, 0

    for start in range(0, len(users), batch_size):
        batch = users[start:start + batch_size]
        scores = np.asarray(model.score(batch), dtype=np.float32)

        # mask already-seen items
        seen = X_seen[batch]
        rows = np.repeat(np.arange(len(batch)), np.diff(seen.indptr))
        scores[rows, seen.indices] = -np.inf

        # top-max_k, then sort just those
        part = np.argpartition(-scores, max_k - 1, axis=1)[:, :max_k]
        r = np.arange(len(batch))[:, None]
        order = np.argsort(-scores[r, part], axis=1)
        topk = part[r, order]                              # (B, max_k)

        truth = X_truth[batch]
        truth_dense = np.zeros((len(batch), N_ITEMS), dtype=bool)
        trows = np.repeat(np.arange(len(batch)), np.diff(truth.indptr))
        truth_dense[trows, truth.indices] = True

        hits = truth_dense[r, topk]                        # (B, max_k) bool
        n_rel = truth_dense.sum(axis=1)

        if collect_items:
            recommended[np.unique(topk[:, :PRIMARY_K])] = True
            novelty_sum += -np.log2(item_pop_prob[topk[:, :PRIMARY_K]]).sum()
            novelty_n += topk[:, :PRIMARY_K].size

        for k in ks:
            hk = hits[:, :k]
            n_hit = hk.sum(axis=1)
            acc[k]["recall"].append(n_hit / np.maximum(n_rel, 1))
            acc[k]["precision"].append(n_hit / k)
            acc[k]["hit"].append((n_hit > 0).astype(np.float32))

            dcg = (hk * _dcg_discounts(k)).sum(axis=1)
            idcg = _idcg_table(k)[np.minimum(n_rel, k) - 1]
            acc[k]["ndcg"].append(dcg / np.maximum(idcg, 1e-12))

        del scores, truth_dense
    
    out = {}
    for k in ks:
        for name, chunks in acc[k].items():
            out[f"{name}@{k}"] = float(np.concatenate(chunks).mean())
    if collect_items:
        out["coverage"] = float(recommended.mean())
        out["novelty"] = float(novelty_sum / max(novelty_n, 1))
    out["_per_user_ndcg"] = np.concatenate(acc[PRIMARY_K]["ndcg"])
    return out


def bootstrap_ci(values, n_boot=1000, alpha=0.05):
    """Percentile CI on the mean. Reported so 0.16 vs 0.21 can be judged."""
    values = np.asarray(values)
    idx = rng.integers(0, len(values), size=(n_boot, len(values)))
    means = values[idx].mean(axis=1)
    return float(np.percentile(means, 100 * alpha / 2)), float(np.percentile(means, 100 * (1 - alpha / 2)))


def report(name, metrics):
    """One-line summary plus a bootstrap CI, so small differences can be judged."""
    lo, hi = bootstrap_ci(metrics["_per_user_ndcg"])
    parts = []
    for k in (PRIMARY_K, 20):
        if f"recall@{k}" in metrics:
            parts.append(f"Recall@{k}={metrics[f'recall@{k}']:.4f}")
            parts.append(f"NDCG@{k}={metrics[f'ndcg@{k}']:.4f}")
    parts.append(f"HR@{PRIMARY_K}={metrics[f'hit@{PRIMARY_K}']:.4f}")
    parts.append(f"P@{PRIMARY_K}={metrics[f'precision@{PRIMARY_K}']:.4f}")
    print(f"\n{name}")
    print("  " + "  ".join(parts))
    tail = f"  NDCG@{PRIMARY_K} 95% CI [{lo:.4f}, {hi:.4f}]"
    if "coverage" in metrics:
        tail += f"  |  coverage={metrics['coverage']:.3f}  novelty={metrics['novelty']:.2f}"
    print(tail)


RESULTS = {}
print("harness ready")

## 6. Baselines

Without these, an NDCG@10 of 0.10 means nothing — it could be worse than
recommending the ten most popular films to everyone. Run them first, and every
later number has something to beat.

**Popularity** is non-personalised: the same ranking for every user, minus what
they have already seen. On MovieLens this is a deceptively strong baseline.

**ItemKNN** is the corrected version of the previous notebook's collaborative
component: cosine similarity between item columns with shrinkage, neighbours
pruned to the top 200, user scores by *summing* over their history rather than
taking the max. Summing matters — max discards the evidence that several of a
user's films point to the same recommendation.

**PureSVD** truncates the interaction matrix to 256 latent factors and
reconstructs. Cheap, and it establishes where plain matrix factorisation lands.

In [ ]:
class PopularityModel:
    name = "Popularity"

    def fit(self, X):
        self.scores_ = np.asarray(X.sum(axis=0)).ravel().astype(np.float32)
        return self

    def score(self, users):
        return np.tile(self.scores_, (len(users), 1))


with timed("Popularity"):
    pop_model = PopularityModel().fit(X_train)

RESULTS["Popularity"] = evaluate(pop_model, eval_users, X_train, X_test)
report("Popularity", RESULTS["Popularity"])

In [ ]:
class ItemKNN:
    """Cosine item-item similarity with shrinkage, top-k neighbours retained.

    Shrinkage pulls similarities toward zero when two items share few users,
    which is what stops a pair of obscure films that happen to share three
    raters from looking like perfect neighbours.
    """
    name = "ItemKNN"

    def __init__(self, k=ITEMKNN_K, shrink=ITEMKNN_SHRINK, chunk=512):
        self.k, self.shrink, self.chunk = k, shrink, chunk

    def fit(self, X):
        Xc = X.tocsc().astype(np.float32)
        norms = np.sqrt(np.asarray(Xc.multiply(Xc).sum(axis=0)).ravel())
        n_items = X.shape[1]

        rows, cols, vals = [], [], []
        for start in range(0, n_items, self.chunk):
            end = min(start + self.chunk, n_items)
            # co-occurrence counts between this chunk and every item
            block = (Xc[:, start:end].T @ Xc).toarray().astype(np.float32, copy=False)

            denom = norms[start:end, None] * norms[None, :] + self.shrink + 1e-8
            block /= denom
            block[np.arange(end - start), np.arange(start, end)] = 0.0   # no self-similarity

            kk = min(self.k, n_items - 1)
            idx = np.argpartition(-block, kk - 1, axis=1)[:, :kk]
            r = np.arange(end - start)[:, None]
            v = block[r, idx]
            keep = v > 0

            rows.append((np.repeat(np.arange(start, end), kk))[keep.ravel()])
            cols.append(idx.ravel()[keep.ravel()])
            vals.append(v.ravel()[keep.ravel()])
            del block

        self.S_ = sp.csr_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(n_items, n_items), dtype=np.float32,
        )
        print(f"    similarity nnz {self.S_.nnz:,} "
              f"({self.S_.nnz / n_items:.0f} neighbours/item)")
        return self

    def score(self, users):
        # sum of similarities from every item in the user's history
        return (X_train[users] @ self.S_).toarray().astype(np.float32, copy=False)


with timed("ItemKNN fit"):
    knn_model = ItemKNN().fit(X_train)

RESULTS["ItemKNN"] = evaluate(knn_model, eval_users, X_train, X_test)
report("ItemKNN", RESULTS["ItemKNN"])

In [ ]:
class PureSVD:
    """Truncated SVD of the binary matrix; scores = X V Vᵀ."""
    name = "PureSVD"

    def __init__(self, factors=SVD_FACTORS):
        self.factors = factors

    def fit(self, X):
        _, _, vt = svds(X.astype(np.float32), k=self.factors)
        self.V_ = np.ascontiguousarray(vt.T, dtype=np.float32)    # (n_items, f)
        return self

    def score(self, users):
        U = X_train[users].toarray().astype(np.float32, copy=False)
        return (U @ self.V_) @ self.V_.T


with timed(f"PureSVD ({SVD_FACTORS} factors)"):
    svd_model = PureSVD().fit(X_train)

RESULTS["PureSVD"] = evaluate(svd_model, eval_users, X_train, X_test)
report("PureSVD", RESULTS["PureSVD"])

## 7. EASE

*Embarrassingly Shallow Autoencoder* (Steck, WWW 2019). A linear item-item model
whose weight matrix has a closed-form solution — no gradient descent, no epochs,
no learning rate.

It solves

$$\min_B \; \lVert X - XB \rVert_F^2 + \lambda \lVert B \rVert_F^2
\quad \text{subject to} \quad \mathrm{diag}(B) = 0$$

The zero-diagonal constraint is the whole trick. Without it the optimum is
$B = I$ — every item perfectly predicts itself and the model learns nothing.
Constraining it away forces each item to be reconstructed from *other* items,
which is exactly the recommendation signal.

The solution, via a Lagrange multiplier per diagonal element:

$$P = (X^\top X + \lambda I)^{-1}, \qquad
B = I - P \cdot \mathrm{diagMat}\!\left(\tfrac{1}{\mathrm{diag}(P)}\right)$$

Why it beats the item-item cosine from the previous notebook: cosine treats each
pair independently, so a user's history double-counts correlated items — rate
three Lord of the Rings films and the model becomes very confident about epic
fantasy and nothing else. The matrix inverse accounts for all items
*jointly*, effectively discounting redundant evidence.

Cost is $O(n_{items}^3)$ for the inverse and independent of the number of users,
which is why all 25M ratings can be used. At 15,000 items expect a few minutes
and roughly 4 GB peak.

In [ ]:
class EASE:
    name = "EASE"

    def __init__(self, lam=250.0):
        self.lam = lam

    def fit(self, X):
        n_items = X.shape[1]

        # Gram matrix. float64 for a well-conditioned inverse: the condition
        # number of XᵀX + λI is high enough that float32 inversion visibly
        # degrades the top of the ranking.
        G = (X.T @ X).toarray().astype(np.float64)
        diag = np.diag_indices(n_items)
        G[diag] += self.lam

        P = np.linalg.inv(G)
        del G
        gc.collect()

        B = P / (-np.diag(P))
        B[diag] = 0.0

        self.B_ = np.ascontiguousarray(B, dtype=np.float32)
        del P, B
        gc.collect()

        print(f"    B: {self.B_.shape}, {mem_mb(self.B_):.0f} MB float32")
        return self

    def score(self, users):
        return np.asarray((X_train[users] @ self.B_), dtype=np.float32)


# --- sweep lambda on the VALIDATION split -------------------------------
# Fitting once per lambda is the honest way to do this; each fit is one
# inverse. If time is tight, fit at the middle value and move on.
sweep = []
best_lam, best_ndcg, best_model = None, -1.0, None

for lam in EASE_LAMBDAS:
    with timed(f"EASE lambda={lam:g}"):
        m = EASE(lam=lam).fit(X_train)
    v = evaluate(m, tune_users, X_train, X_val, ks=(PRIMARY_K,), collect_items=False)
    score = v[f"ndcg@{PRIMARY_K}"]
    sweep.append({"lambda": lam, f"val_ndcg@{PRIMARY_K}": score})
    print(f"    val NDCG@{PRIMARY_K} = {score:.4f}")
    if score > best_ndcg:
        best_ndcg, best_lam, best_model = score, lam, m
    else:
        del m
        gc.collect()

sweep_df = pd.DataFrame(sweep)
print(f"\n  best lambda = {best_lam:g} (val NDCG@{PRIMARY_K} = {best_ndcg:.4f})")
sweep_df

In [ ]:
ease_model = best_model
RESULTS["EASE"] = evaluate(ease_model, eval_users, X_train, X_test)
report(f"EASE (lambda={best_lam:g})", RESULTS["EASE"])

## 8. Content model

The content side earns its place in two situations, neither of which is
"beating EASE on warm users":

1. **Cold-start items** — a film nobody has rated yet has no column in the CF
   model, so it can never be recommended. Content similarity gives it one.
2. **Tail coverage** — content pulls in relevant-but-unpopular items that CF's
   popularity bias buries.

Two changes from the previous approach. Tags are capped at the 40 most frequent
per film: the original concatenated *every* tag a film ever received, so popular
films got paragraphs of text and obscure ones got three words, and the encoder
mostly learned to measure popularity. And IDF weighting is applied so that
"Drama" — present on a third of the catalogue — carries less weight than
"neo-noir".

TF-IDF is the default because it runs in seconds on CPU and lands within a few
percent of the sentence encoder here. Flip `USE_CONTENT_SBERT` for the encoder.

In [ ]:
def build_item_text(tags: pd.DataFrame, meta: pd.DataFrame) -> pd.Series:
    """One text document per item, aligned to item_idx."""
    t = tags.dropna(subset=["movieId", "tag"]).copy()
    t["tag"] = t["tag"].astype(str).str.strip().str.lower()
    t = t[(t["tag"] != "") & t["movieId"].isin(item_to_idx)]

    # keep only the most frequent tags per film so text length does not
    # simply encode popularity
    counts = t.groupby(["movieId", "tag"]).size().rename("n").reset_index()
    counts = counts.sort_values(["movieId", "n"], ascending=[True, False])
    counts = counts.groupby("movieId").head(MAX_TAGS_PER_ITEM)
    tag_text = counts.groupby("movieId")["tag"].apply(" ".join)

    year = meta["title"].str.extract(r"\((\d{4})\)")[0].fillna("")
    decade = year.where(year == "", (year.str[:3] + "0s"))

    genres = meta["genres"].str.replace("|", " ", regex=False).str.lower()
    genres = genres.replace("(no genres listed)", "")

    tags_aligned = meta["movieId"].map(tag_text).fillna("")

    text = (genres + " " + decade + " " + tags_aligned).str.strip()
    print(f"  {(text == '').sum():,} items with empty text "
          f"({(text == '').mean():.1%})")
    print(f"  median text length: {text.str.split().str.len().median():.0f} tokens")
    return text


item_text = build_item_text(tags_raw, item_meta)
item_meta["text"] = item_text
item_meta[["movieId", "title", "genres", "text"]].head(5)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize

def build_content_embeddings(text: pd.Series) -> np.ndarray:
    """Return L2-normalised item embeddings, row i == item_idx i."""
    if USE_CONTENT_SBERT:
        from sentence_transformers import SentenceTransformer
        import torch
        device = "cuda" if torch.cuda.is_available() else "cpu"
        enc = SentenceTransformer(SBERT_MODEL, device=device)
        emb = enc.encode(
            text.tolist(), batch_size=256, convert_to_numpy=True,
            normalize_embeddings=True, show_progress_bar=True,
        )
        return emb.astype(np.float32)

    global _tfidf_vec
    _tfidf_vec = TfidfVectorizer(
        min_df=3, max_features=60_000, sublinear_tf=True, ngram_range=(1, 2)
    )
    M = _tfidf_vec.fit_transform(text.tolist())
    M = normalize(M, norm="l2", axis=1)
    print(f"  TF-IDF vocabulary: {len(_tfidf_vec.vocabulary_):,}")
    return M.astype(np.float32)          # kept sparse


with timed("content embeddings"):
    content_emb = build_content_embeddings(item_text)

print(f"  shape {content_emb.shape}")
assert content_emb.shape[0] == N_ITEMS, "content rows must align with item_idx"

In [ ]:
def content_knn(emb, k=CONTENT_KNN_K, chunk=512) -> sp.csr_matrix:
    """Top-k cosine neighbours per item. Rows are L2-normalised already,
    so the dot product is the cosine."""
    n = emb.shape[0]
    rows, cols, vals = [], [], []
    for start in range(0, n, chunk):
        end = min(start + chunk, n)
        block = emb[start:end] @ emb.T
        block = (block.toarray() if sp.issparse(block) else np.asarray(block))
        block = block.astype(np.float32, copy=False)
        block[np.arange(end - start), np.arange(start, end)] = 0.0

        kk = min(k, n - 1)
        idx = np.argpartition(-block, kk - 1, axis=1)[:, :kk]
        r = np.arange(end - start)[:, None]
        v = block[r, idx]
        keep = (v > 1e-6).ravel()

        rows.append(np.repeat(np.arange(start, end), kk)[keep])
        cols.append(idx.ravel()[keep])
        vals.append(v.ravel()[keep])
        del block

    S = sp.csr_matrix(
        (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
        shape=(n, n), dtype=np.float32,
    )
    print(f"    nnz {S.nnz:,} ({S.nnz / n:.0f} neighbours/item)")
    return S


class ContentModel:
    name = "Content"

    def __init__(self, S):
        self.S_ = S

    def score(self, users):
        return (X_train[users] @ self.S_).toarray().astype(np.float32, copy=False)


with timed("content item-item kNN"):
    S_content = content_knn(content_emb)

content_model = ContentModel(S_content)
RESULTS["Content"] = evaluate(content_model, eval_users, X_train, X_test)
report("Content only", RESULTS["Content"])

## 9. Hybrid

The previous notebook blended by min-max normalising each candidate list, which
forces the best candidate to exactly 1.0 and the worst to exactly 0.0 *every
time*. A user whose top content match was mediocre and a user whose top match was
perfect both got a 1.0, so the weight sweep was comparing rescaled noise — hence
the suspiciously flat Hit Rate curve.

Here both score vectors are **z-scored per user across the full item set**. That
preserves the shape of each distribution: a confident CF score stays large
relative to its own mean, a diffuse one does not. Only then does the weight mean
something.

`alpha` is tuned on validation, never on test.

In [ ]:
def zscore_rows(A: np.ndarray) -> np.ndarray:
    mu = A.mean(axis=1, keepdims=True)
    sd = A.std(axis=1, keepdims=True)
    return (A - mu) / np.maximum(sd, 1e-8)


class HybridModel:
    name = "Hybrid"

    def __init__(self, cf, content, alpha=0.9):
        self.cf, self.content, self.alpha = cf, content, alpha

    def score(self, users):
        a = zscore_rows(self.cf.score(users))
        if self.alpha >= 1.0:
            return a
        b = zscore_rows(self.content.score(users))
        return self.alpha * a + (1.0 - self.alpha) * b


rows = []
best_alpha, best_val = None, -1.0
for alpha in HYBRID_ALPHAS:
    h = HybridModel(ease_model, content_model, alpha=alpha)
    v = evaluate(h, tune_users, X_train, X_val, ks=(PRIMARY_K,), collect_items=False)
    s = v[f"ndcg@{PRIMARY_K}"]
    rows.append({"alpha (CF weight)": alpha, f"val_ndcg@{PRIMARY_K}": s})
    print(f"  alpha={alpha:<5} val NDCG@{PRIMARY_K}={s:.4f}")
    if s > best_val:
        best_val, best_alpha = s, alpha

alpha_df = pd.DataFrame(rows)
print(f"\n  best alpha = {best_alpha}")
alpha_df

In [ ]:
hybrid_model = HybridModel(ease_model, content_model, alpha=best_alpha)
RESULTS["Hybrid"] = evaluate(hybrid_model, eval_users, X_train, X_test)
report(f"Hybrid (alpha={best_alpha})", RESULTS["Hybrid"])

## 10. Results

Read `Recall@20` and `NDCG@10` as the headline numbers. `Precision@10` is
capped at 0.5 here because only 5 items are held out per user, so it is
reported for continuity with the previous notebook rather than as the metric
to optimise.

If the hybrid's best `alpha` came out at or near 1.0, that is a real finding,
not a failure: it says the content signal adds nothing *for users who already
have history*, which is what the literature generally reports on MovieLens.
Its value shows up in the cold-start section below.

In [ ]:
summary = pd.DataFrame({
    name: {
        f"Recall@{PRIMARY_K}": m[f"recall@{PRIMARY_K}"],
        "Recall@20": m["recall@20"],
        f"NDCG@{PRIMARY_K}": m[f"ndcg@{PRIMARY_K}"],
        "NDCG@20": m["ndcg@20"],
        f"Precision@{PRIMARY_K}": m[f"precision@{PRIMARY_K}"],
        f"HitRate@{PRIMARY_K}": m[f"hit@{PRIMARY_K}"],
        "Coverage": m.get("coverage", np.nan),
        "Novelty": m.get("novelty", np.nan),
    }
    for name, m in RESULTS.items()
}).T

summary = summary.sort_values(f"NDCG@{PRIMARY_K}", ascending=False)
summary.round(4)

In [ ]:
# Lift over the popularity baseline -- the number worth putting in a write-up
base = summary.loc["Popularity", f"NDCG@{PRIMARY_K}"]
lift = ((summary[f"NDCG@{PRIMARY_K}"] / base - 1) * 100).round(1)
print(f"NDCG@{PRIMARY_K} lift over the popularity baseline")
for name, v in lift.items():
    print(f"  {name:<12} {v:+7.1f}%")

summary.to_csv(OUT_DIR / "results_summary.csv")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
order = summary.index.tolist()

for ax, col, title in zip(
    axes,
    [f"NDCG@{PRIMARY_K}", "Recall@20", "Coverage"],
    [f"NDCG@{PRIMARY_K} (higher is better)", "Recall@20", "Catalogue coverage"],
):
    vals = summary[col].values
    ax.barh(order, vals, color="#3b6ea5")
    ax.invert_yaxis()
    ax.set_title(title, fontsize=11)
    ax.grid(axis="x", alpha=0.3)
    for i, v in enumerate(vals):
        if not np.isnan(v):
            ax.text(v, i, f" {v:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig(OUT_DIR / "model_comparison.png", dpi=140, bbox_inches="tight")
plt.show()

In [ ]:
# Where does each model win or lose? Bucket users by how much history they have.
def performance_by_history(model, users, buckets=((5, 20), (20, 50), (50, 150), (150, 10**9))):
    hist = np.diff(X_train.indptr)
    out = []
    for lo, hi in buckets:
        sel = users[(hist[users] >= lo) & (hist[users] < hi)]
        if len(sel) < 50:
            continue
        m = evaluate(model, sel, X_train, X_test, ks=(PRIMARY_K,), collect_items=False)
        out.append({
            "history": f"{lo}-{hi if hi < 10**8 else '+'}",
            "users": len(sel),
            f"ndcg@{PRIMARY_K}": round(m[f"ndcg@{PRIMARY_K}"], 4),
        })
    return pd.DataFrame(out)


cold_analysis = performance_by_history(ease_model, eval_users).merge(
    performance_by_history(content_model, eval_users),
    on=["history", "users"], suffixes=("_ease", "_content"),
)
print("Sparse-history users are where the content signal should help:")
display(cold_analysis)

## 11. Inference

Everything below takes `movieId` / `userId` and converts through the mappings —
no title string matching anywhere, which is what silently corrupted the previous
notebook's evaluation.

In [ ]:
def _titles(item_indices, extra=None):
    out = item_meta.iloc[item_indices][["movieId", "title", "genres"]].reset_index(drop=True)
    if extra is not None:
        for col, vals in extra.items():
            out[col] = vals
    return out


def recommend_for_user(user_id, k=10, model=None, exclude_seen=True):
    """Top-k for an existing user."""
    model = model or hybrid_model
    if user_id not in user_to_idx:
        raise KeyError(f"unknown userId {user_id}; use recommend_from_items() instead")

    u = user_to_idx[user_id]
    scores = model.score(np.array([u]))[0]
    if exclude_seen:
        scores[X_train[u].indices] = -np.inf

    top = np.argpartition(-scores, k)[:k]
    top = top[np.argsort(-scores[top])]
    return _titles(top, {"score": scores[top]})


def similar_items(movie_id, k=10, use="cf"):
    """Nearest neighbours of an item. use='cf' | 'content'."""
    if movie_id not in item_to_idx:
        raise KeyError(f"unknown movieId {movie_id}")
    j = item_to_idx[movie_id]

    if use == "cf":
        sims = ease_model.B_[:, j].copy()       # column: how much j is predicted by others
    else:
        sims = S_content[j].toarray().ravel()

    sims[j] = -np.inf
    top = np.argpartition(-sims, k)[:k]
    top = top[np.argsort(-sims[top])]
    return _titles(top, {"similarity": sims[top]})


def recommend_from_items(liked_movie_ids, k=10, alpha=None):
    """Cold-start: a brand-new user described only by a few films they like."""
    alpha = best_alpha if alpha is None else alpha
    idx = [item_to_idx[m] for m in liked_movie_ids if m in item_to_idx]
    if not idx:
        raise ValueError("none of those movieIds are in the trained catalogue")

    profile = sp.csr_matrix(
        (np.ones(len(idx), dtype=np.float32), (np.zeros(len(idx)), idx)),
        shape=(1, N_ITEMS), dtype=np.float32,
    )
    cf = zscore_rows(np.asarray(profile @ ease_model.B_, dtype=np.float32))
    ct = zscore_rows((profile @ S_content).toarray().astype(np.float32, copy=False))
    scores = (alpha * cf + (1 - alpha) * ct)[0]
    scores[idx] = -np.inf

    top = np.argpartition(-scores, k)[:k]
    top = top[np.argsort(-scores[top])]
    return _titles(top, {"score": scores[top]})


def _pick(preferred_ids):
    """Fall back to a popular in-catalogue item if the preferred ids were
    filtered out by the k-core or the item cap."""
    for m in preferred_ids:
        if m in item_to_idx:
            return m
    return int(idx_to_item[int(np.argmax(item_pop))])


TOY_STORY = _pick([1])          # movieId 1 == Toy Story (1995)
_name = item_meta.iloc[item_to_idx[TOY_STORY]]["title"]

print(f"Similar to {_name} — collaborative (EASE):")
display(similar_items(TOY_STORY, k=10, use="cf"))
print(f"\nSimilar to {_name} — content:")
display(similar_items(TOY_STORY, k=10, use="content"))

In [ ]:
# A brand-new user, described only by three films they like:
#   2571 The Matrix, 79132 Inception, 2959 Fight Club
cold_user_likes = [m for m in (2571, 79132, 2959) if m in item_to_idx]
if not cold_user_likes:                       # tiny catalogue / heavy filtering
    cold_user_likes = [int(idx_to_item[i]) for i in np.argsort(-item_pop)[:3]]

print("Seed films:")
display(item_meta.iloc[[item_to_idx[m] for m in cold_user_likes]][["movieId", "title", "genres"]])
print("\nCold-start recommendations:")
recommend_from_items(cold_user_likes, k=10)

In [ ]:
# Warm user, side by side with what they actually went on to watch
demo_user_idx = eval_users[0]
demo_user_id = int(idx_to_user[demo_user_idx])

history = item_meta.iloc[X_train[demo_user_idx].indices[-8:]][["title", "genres"]]
truth = item_meta.iloc[X_test[demo_user_idx].indices][["movieId", "title"]]

print(f"userId {demo_user_id} — recent history:")
display(history)
print("held-out (what they actually liked next):")
display(truth)
print("recommended:")
display(recommend_for_user(demo_user_id, k=10))

## 12. Cold-start items

An item with no training interactions has an all-zero column in `B`, so EASE can
never surface it. The fix is to borrow a CF representation from its content
neighbours: a new film's predicted column is the interaction-weighted average of
the columns of the films it most resembles textually.

This is the concrete answer to "why keep the content model at all when EASE wins
on warm users" — and it is a good thing to be able to point at in an interview,
because cold start is the failure mode every production recommender actually
fights.

In [ ]:
def cold_item_column(text: str, n_neighbours=25) -> np.ndarray:
    """Approximate an unseen item's EASE column from its content neighbours."""
    if USE_CONTENT_SBERT:
        raise NotImplementedError("re-encode the text with the SBERT model here")

    q = _tfidf_vec.transform([text])          # the vectoriser fitted above
    q = normalize(q, norm="l2", axis=1)
    sims = (q @ content_emb.T).toarray().ravel()

    top = np.argpartition(-sims, n_neighbours)[:n_neighbours]
    w = sims[top]
    if w.sum() <= 0:
        return np.zeros(N_ITEMS, dtype=np.float32)
    w = w / w.sum()
    return (ease_model.B_[:, top] * w).sum(axis=1).astype(np.float32)


if not USE_CONTENT_SBERT:
    new_film_text = "sci-fi thriller 2020s time travel dystopia cerebral twist ending"
    col = cold_item_column(new_film_text)
    print("A hypothetical new film described as:")
    print(f"  {new_film_text!r}")
    print(f"\nnon-zero weight against {np.count_nonzero(col):,} catalogue items")
    print("it would be recommended most strongly to users who liked:")
    top = np.argsort(-col)[:8]
    display(_titles(top, {"weight": col[top]}))

## 13. Save

`B` is the model. At 15,000 items it is roughly 900 MB in float32 — large for a
repository, so in production you would sparsify it (keeping the top ~500 weights
per column costs very little accuracy) or serve scores from an approximate
nearest-neighbour index.

In [ ]:
def sparsify(B: np.ndarray, keep_per_col=500) -> sp.csr_matrix:
    """Keep the largest `keep_per_col` weights per column. Usually costs <1% NDCG
    and shrinks the artifact by an order of magnitude."""
    n = B.shape[0]
    out = np.zeros_like(B)
    for start in range(0, n, 1024):
        end = min(start + 1024, n)
        block = B[:, start:end]
        kk = min(keep_per_col, n - 1)
        idx = np.argpartition(-block, kk - 1, axis=0)[:kk]
        c = np.arange(end - start)[None, :]
        out[idx, c + start] = block[idx, c]
    return sp.csr_matrix(out)


B_sparse = sparsify(ease_model.B_)
print(f"  dense  {ease_model.B_.nbytes / 1e6:>8.0f} MB")
print(f"  sparse {(B_sparse.data.nbytes + B_sparse.indices.nbytes + B_sparse.indptr.nbytes) / 1e6:>8.0f} MB")

# confirm the compression did not cost accuracy
class SparseEASE:
    def __init__(self, B): self.B_ = B
    def score(self, users): return (X_train[users] @ self.B_).toarray().astype(np.float32, copy=False)

m = evaluate(SparseEASE(B_sparse), eval_users[:3000], X_train, X_test,
             ks=(PRIMARY_K,), collect_items=False)
print(f"\n  dense  NDCG@{PRIMARY_K} = {RESULTS['EASE'][f'ndcg@{PRIMARY_K}']:.4f}")
print(f"  sparse NDCG@{PRIMARY_K} = {m[f'ndcg@{PRIMARY_K}']:.4f}")

In [ ]:
sp.save_npz(OUT_DIR / "ease_B_sparse.npz", B_sparse)
sp.save_npz(OUT_DIR / "content_knn.npz", S_content)
np.save(OUT_DIR / "item_ids.npy", idx_to_item)
np.save(OUT_DIR / "user_ids.npy", idx_to_user)
item_meta[["movieId", "title", "genres", "item_idx"]].to_csv(OUT_DIR / "item_meta.csv", index=False)

import json
json.dump(
    {
        "ease_lambda": best_lam,
        "hybrid_alpha": best_alpha,
        "n_users": int(N_USERS),
        "n_items": int(N_ITEMS),
        "positive_threshold": POSITIVE_THRESHOLD,
        "test_n_per_user": TEST_N_PER_USER,
        "metrics": {k: {kk: vv for kk, vv in v.items() if not kk.startswith("_")}
                    for k, v in RESULTS.items()},
    },
    open(OUT_DIR / "model_config.json", "w"), indent=2,
)
print("saved:", *[p.name for p in sorted(OUT_DIR.glob('*')) if p.is_file()], sep="\n  ")

## 14. Where to go next

Roughly in order of return on effort.

**Multi-VAE / RecVAE.** The main non-linear competitor to EASE on this dataset.
Sometimes wins on Recall@50 while losing on NDCG@10. A genuine step up in
modelling sophistication if you want a deep-learning result on the CV.

**Sequential models (SASRec, BERT4Rec).** EASE ignores order entirely — it sees a
user's history as a set. Given that the split here is chronological, a
self-attention model over the sequence is the natural next step, and it connects
directly to the Transformer work already on your resume.

**Strong generalisation.** The protocol here evaluates users who appear in
training. Holding out entire *users* — fold in their history at inference time,
predict their held-out items — is the harder and more realistic setting, and the
one the Multi-VAE paper uses. Numbers will drop; that is expected.

**Popularity debiasing.** Check whether the coverage figure is acceptable. If the
top-10 lists are dominated by 500 films out of 15,000, inverse-propensity
reweighting during training is the standard remedy.

**A/B-relevant metrics.** Offline NDCG correlates imperfectly with engagement.
Serendipity and intra-list diversity are cheap to add and show you know the
offline/online gap exists.

### On presenting this

The result that reads best is not "NDCG went from 0.10 to 0.35". It is:

> Diagnosed that an inner-join between candidate generators was collapsing the
> recommendation pool, established a popularity baseline that the original model
> had been losing to, and replaced item-item cosine with a closed-form linear
> autoencoder (EASE) — NDCG@10 0.10 → 0.35 on MovieLens-25M, evaluated over
> 10,000 users with bootstrap confidence intervals.

The debugging is the part that distinguishes you. Most people can import a
better model; far fewer notice that their evaluation was measuring the wrong
thing.